In [1]:
import torch
import matplotlib.pyplot as plt

torch.set_printoptions(precision=2, sci_mode=False)

In [2]:
def no_randomness():
    torch.manual_seed(31)

In [3]:
no_randomness()
inputs = torch.randn(1, 2, 9)
inputs

tensor([[[-0.18, -0.32,  0.38,  1.12, -0.04,  1.01, -0.17,  2.09, -0.32],
         [ 1.10, -0.41, -1.12,  2.09, -1.28, -2.11, -0.20,  0.47, -0.91]]])

In [4]:
no_randomness()


class LayerNorm(torch.nn.Module):
    def __init__(self, d_out):
        super().__init__()
        self.eps = 1e-5
        self.scale = torch.nn.Parameter(torch.ones(d_out))
        self.shift = torch.nn.Parameter(torch.zeros(d_out))

    def forward(self, x):
        # x = (x - x.mean(dim=-1, keepdims=True)) / x.std(dim=-1, keepdims=True)
        x = (x - x.mean(dim=-1, keepdims=True)) / (
            torch.sqrt(x.var(dim=-1, keepdims=True, correction=0) + self.eps)
        )
        return self.scale * x + self.shift


print(inputs.mean(dim=-1), inputs.std(dim=-1))
ln_out = LayerNorm(inputs.shape[-1])(inputs)
print(ln_out.mean(dim=-1), ln_out.std(dim=-1, correction=0))

tensor([[ 0.40, -0.26]]) tensor([[0.84, 1.30]])
tensor([[     0.00,     -0.00]], grad_fn=<MeanBackward1>) tensor([[1.00, 1.00]], grad_fn=<StdBackward0>)


In [5]:
no_randomness()


class FeedForwardNetwork(torch.nn.Module):

    def __init__(self, hidden_dim):
        super().__init__()
        self.layers = torch.nn.Sequential(
            torch.nn.Linear(hidden_dim, hidden_dim),
            torch.nn.GELU(),
            torch.nn.Linear(hidden_dim, hidden_dim),
        )

    def forward(self, x):
        return self.layers(x)


print(inputs)
print(FeedForwardNetwork(inputs.shape[-1])(inputs))

tensor([[[-0.18, -0.32,  0.38,  1.12, -0.04,  1.01, -0.17,  2.09, -0.32],
         [ 1.10, -0.41, -1.12,  2.09, -1.28, -2.11, -0.20,  0.47, -0.91]]])
tensor([[[ 0.02,  0.11, -0.23,  0.25,  0.23, -0.44, -0.42, -0.21, -0.22],
         [-0.32,  0.19,  0.31,  0.37,  0.30, -0.23, -0.00,  0.19,  0.02]]],
       grad_fn=<ViewBackward0>)


In [6]:
no_randomness()


class MultiHeadAttention(torch.nn.Module):

    def __init__(
        self,
        emb_size,
        hidden_dim,
        num_heads,
        context_len,
        dropout_p=0.2,
        qkv_bias=False,
    ):
        super().__init__()
        assert (
            hidden_dim % num_heads == 0
        ), "Number of Heads should be divisible to the hidden dims"
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads
        # print(self.num_heads, self.head_dim)
        self.WQ = torch.nn.Linear(emb_size, hidden_dim, bias=qkv_bias)
        self.WK = torch.nn.Linear(emb_size, hidden_dim, bias=qkv_bias)
        self.WV = torch.nn.Linear(emb_size, hidden_dim, bias=qkv_bias)
        self.DO = torch.nn.Dropout(dropout_p)
        self.register_buffer(
            "mask", torch.triu(torch.ones(context_len, context_len), diagonal=1)
        )

    def forward(self, x):
        batch_size, num_tokens, emb_size = x.shape
        Q = self.WQ(x)
        K = self.WK(x)
        V = self.WV(x)
        # print(Q.shape, K.shape, V.shape)

        # add heads
        Qh = Q.view(batch_size, num_tokens, self.num_heads, self.head_dim).transpose(
            1, 2
        )
        Kh = K.view(batch_size, num_tokens, self.num_heads, self.head_dim).transpose(
            1, 2
        )
        Vh = V.view(batch_size, num_tokens, self.num_heads, self.head_dim).transpose(
            1, 2
        )

        # print(Qh.shape, Kh.shape, Vh.shape)

        atten_score = Qh @ Kh.transpose(2, 3)
        # print(atten_score.shape)
        atten_score.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        atten_score = self.DO(torch.softmax(atten_score, dim=-1))
        CTXh = (atten_score @ Vh).transpose(1, 2)
        CTX = CTXh.contiguous().view(batch_size, num_tokens, emb_size)
        return CTX


mha = MultiHeadAttention(
    emb_size=inputs.shape[-1],
    hidden_dim=inputs.shape[-1],
    num_heads=3,
    context_len=128,
    dropout_p=0,
)(inputs)
mha_ln = LayerNorm(inputs.shape[-1])(mha)

In [10]:
no_randomness()


class Transformer(torch.nn.Module):

    def __init__(
        self,
        emb_size,
        hidden_dim,
        num_heads,
        context_len,
        dropout_p,
    ):
        super().__init__()
        self.layer_norm1 = LayerNorm(emb_size)
        self.layer_norm2 = LayerNorm(emb_size)
        self.mha = MultiHeadAttention(
            emb_size=emb_size,
            hidden_dim=hidden_dim,
            num_heads=num_heads,
            context_len=context_len,
            dropout_p=dropout_p,
        )
        self.drop_out = torch.nn.Dropout(dropout_p)
        self.feed_forward = FeedForwardNetwork(hidden_dim)

    def forward(self, x):
        shortcut = x
        x = self.layer_norm1(x)
        x = self.mha(x)
        x = self.drop_out(x)
        x += shortcut

        shortcut = x
        x = self.layer_norm2(x)
        x = self.feed_forward(x)
        x = self.drop_out(x)
        x += shortcut
        return x


transformer = Transformer(
    emb_size=inputs.shape[-1],
    hidden_dim=inputs.shape[-1],
    num_heads=3,
    context_len=128,
    dropout_p=0.2,
)
print(inputs.shape)
print(transformer(inputs).shape)

torch.Size([1, 2, 9])
torch.Size([1, 2, 9])


In [11]:
no_randomness()


class GPTModel(torch.nn.Module):
    def __init__(
        self,
        emb_size,
        vocab_size,
        hidden_dim,
        num_heads,
        context_len,
        num_layers=12,
    ):
        super().__init__()
        self.tok_emb = torch.nn.Embedding(vocab_size, emb_size)
        self.pos_emb = torch.nn.Embedding(context_len, emb_size)
        self.drop_emb = torch.nn.Dropout(0.2)

        self.trf_blocks = torch.nn.Sequential(
            *[
                Transformer(
                    emb_size,
                    hidden_dim,
                    num_heads,
                    context_len,
                    0.2,
                )
                for _ in range(num_layers)
            ]
        )

        self.final_norm = LayerNorm(emb_size)
        self.out_head = torch.nn.Linear(emb_size, vocab_size, bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)

        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits